In [1]:
import datetime
import feed


def get_dates(number_of_days: int = 1):

    end_date = datetime.date.today()
    start_date = end_date - datetime.timedelta(days=number_of_days)

    return int(start_date.strftime("%Y%m%d")), int(end_date.strftime("%Y%m%d"))


symbol = "BTCUSDT"
start_date, end_date = get_dates(90)


dataset = feed.get_data(
    symbol,
    start_date,
    end_date
)

dataset

,symbol,date,time,open,high,low,close,volume
0,BTCUSDT,20260707,0,64042.93,64076.00,64042.93,64056.64,15.85568
1,BTCUSDT,20260707,60,64056.63,64067.67,64043.77,64043.77,11.23046
2,BTCUSDT,20260707,120,64043.78,64044.00,64018.64,64044.00,4.66518
3,BTCUSDT,20260707,180,64044.00,64044.00,64026.00,64026.01,3.64898
4,BTCUSDT,20260707,240,64026.01,64065.19,64024.76,64058.77,61.31728
...,...,...,...,...,...,...,...,...
129595,BTCUSDT,20261004,86100,86478.01,86498.18,86466.00,86468.01,9.00840
129596,BTCUSDT,20261004,86160,86468.01,86509.71,86468.01,86503.56,5.87533
129597,BTCUSDT,20261004,86220,86503.57,86503.57,86480.00,86484.35,2.25165
129598,BTCUSDT,20261004,86280,86484.35,86515.68,86468.00,86512.00,37.30211


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils

In [3]:
dataset["datetime"] = (
    pd.to_datetime(
        dataset["date"].astype(str),
        format="%Y%m%d"
    )
    + pd.to_timedelta(dataset["time"], unit="s")
)
dataset

,symbol,date,time,open,high,low,close,volume,datetime
0,BTCUSDT,20260707,0,64042.93,64076.00,64042.93,64056.64,15.85568,2026-07-07 00:00:00
1,BTCUSDT,20260707,60,64056.63,64067.67,64043.77,64043.77,11.23046,2026-07-07 00:01:00
2,BTCUSDT,20260707,120,64043.78,64044.00,64018.64,64044.00,4.66518,2026-07-07 00:02:00
3,BTCUSDT,20260707,180,64044.00,64044.00,64026.00,64026.01,3.64898,2026-07-07 00:03:00
4,BTCUSDT,20260707,240,64026.01,64065.19,64024.76,64058.77,61.31728,2026-07-07 00:04:00
...,...,...,...,...,...,...,...,...,...
129595,BTCUSDT,20261004,86100,86478.01,86498.18,86466.00,86468.01,9.00840,2026-10-04 23:55:00
129596,BTCUSDT,20261004,86160,86468.01,86509.71,86468.01,86503.56,5.87533,2026-10-04 23:56:00
129597,BTCUSDT,20261004,86220,86503.57,86503.57,86480.00,86484.35,2.25165,2026-10-04 23:57:00
129598,BTCUSDT,20261004,86280,86484.35,86515.68,86468.00,86512.00,37.30211,2026-10-04 23:58:00


In [5]:
import utils


window = 10
source = "close"

for i in range(window - 1, len(dataset)):

    prices = dataset[source].iloc[i-window+1:i+1].tolist()
    times = list(range(window))

    regression = utils.regression_line(prices, times)

    dataset.loc[i, "regression_m"] = regression.m
    dataset.loc[i, "regression_b"] = regression.b
    dataset.loc[i, "r_squared"] = regression.r_squared

dataset = dataset.dropna(subset=["regression_m", "regression_b", "r_squared"])

dataset

,symbol,date,time,open,high,low,close,volume,datetime,regression_m,regression_b,r_squared
9,BTCUSDT,20260707,540,64031.22,64031.23,64030.00,64030.62,1.53318,2026-07-07 00:09:00,0.492061,64051.688727,0.004322
10,BTCUSDT,20260707,600,64030.61,64040.00,64030.00,64040.00,2.58282,2026-07-07 00:10:00,-0.083818,64052.616182,0.000121
11,BTCUSDT,20260707,660,64039.99,64040.00,64021.06,64030.00,1.66375,2026-07-07 00:11:00,-1.861455,64059.238545,0.055141
12,BTCUSDT,20260707,720,64029.99,64045.04,64024.07,64041.13,20.42430,2026-07-07 00:12:00,-2.849758,64063.398909,0.128081
13,BTCUSDT,20260707,780,64041.14,64050.71,64038.76,64050.70,7.16099,2026-07-07 00:13:00,-4.480606,64073.206727,0.362686
...,...,...,...,...,...,...,...,...,...,...,...,...
129595,BTCUSDT,20261004,86100,86478.01,86498.18,86466.00,86468.01,9.00840,2026-10-04 23:55:00,-7.743212,86534.883455,0.542902
129596,BTCUSDT,20261004,86160,86468.01,86509.71,86468.01,86503.56,5.87533,2026-10-04 23:56:00,-6.973091,86530.902909,0.443466
129597,BTCUSDT,20261004,86220,86503.57,86503.57,86480.00,86484.35,2.25165,2026-10-04 23:57:00,-5.769030,86520.919636,0.337364
129598,BTCUSDT,20261004,86280,86484.35,86515.68,86468.00,86512.00,37.30211,2026-10-04 23:58:00,-3.436788,86510.024545,0.122037


In [ ]:
from matplotlib.patches import Rectangle


def plot_regression_window(dataset, end_idx, window=10):

    start_idx = end_idx - window + 1
    data = dataset.iloc[start_idx:end_idx + 1].copy()

    # x positions: 0, 1, 2, ... 9
    x = np.arange(window)

    # --------------------------------------------------
    # Get regression values for THIS 10-candle window
    # --------------------------------------------------

    m = data["regression_m"].iloc[-1]
    b = data["regression_b"].iloc[-1]
    r_squared = data["r_squared"].iloc[-1]

    regression_y = m * x + b

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------

    fig, ax = plt.subplots(figsize=(14, 7))

    candle_width = 0.65

    for i, (_, row) in enumerate(data.iterrows()):

        open_price = row["open"]
        high_price = row["high"]
        low_price = row["low"]
        close_price = row["close"]

        # ----------------------------------------------
        # Candle color
        # ----------------------------------------------

        if close_price >= open_price:
            candle_color = "#26a69a"   # green
        else:
            candle_color = "#ef5350"   # red

        # ----------------------------------------------
        # Wick
        # ----------------------------------------------

        ax.plot(
            [i, i],
            [low_price, high_price],
            color=candle_color,
            linewidth=1.5,
            zorder=1
        )

        # ----------------------------------------------
        # Candle body
        # ----------------------------------------------

        body_low = min(open_price, close_price)
        body_high = max(open_price, close_price)
        body_height = body_high - body_low

        # Do not make doji candles invisible
        if body_height == 0:
            body_height = 0.1

        candle = Rectangle(
            (
                i - candle_width / 2,
                body_low
            ),
            candle_width,
            body_height,
            facecolor=candle_color,
            edgecolor=candle_color,
            linewidth=1,
            zorder=2
        )

        ax.add_patch(candle)

    # --------------------------------------------------
    # Regression line
    # --------------------------------------------------

    ax.plot(
        x,
        regression_y,
        color="blue",
        linewidth=2.5,
        label=f"Regression  m={m:.4f}",
        zorder=3
    )

    # --------------------------------------------------
    # X axis
    # --------------------------------------------------

    ax.set_xticks(x)

    ax.set_xticklabels(
        data["datetime"].dt.strftime("%H:%M"),
        rotation=45
    )

    # --------------------------------------------------
    # Labels
    # --------------------------------------------------

    ax.set_title(
        f"10 Candle Regression\n"
        f"{data['datetime'].iloc[0]} → "
        f"{data['datetime'].iloc[-1]}",
        fontsize=15
    )

    ax.set_xlabel("Time")
    ax.set_ylabel("Price")

    # --------------------------------------------------
    # Grid
    # --------------------------------------------------

    ax.grid(
        True,
        alpha=0.25,
        linestyle="--"
    )

    # --------------------------------------------------
    # Regression equation
    # --------------------------------------------------

    ax.text(
        0.02,
        0.95,
        f"y = {m:.4f}x + {b:.2f}",
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment="top"
    )

    ax.legend()

    plt.tight_layout()
    plt.show()



plot_regression_window(
    dataset,
    end_idx=116,
    window=20
)



#end_index observations 108 - 116 reverse trend

